### **Data Reading**

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

In [0]:
df = spark.read.format("parquet")\
  .load("abfss://bronze@storageaccnitin.dfs.core.windows.net/orders")

In [0]:
df.display()

In [0]:
df.printSchema()

In [0]:
df  = df.withColumnRenamed("_rescued_data","rescued_data")

In [0]:
df = df.drop("rescued_data")

In [0]:
df.display()

In [0]:
df = df.withColumn("order_date", to_timestamp(col("order_date")))
df.display()

In [0]:
df = df.withColumn("year", year(col("order_date")))
df.display()

In [0]:
window = Window.partitionBy(col("year")).orderBy(desc(col("total_amount")))
df1 = df.withColumn("flag", dense_rank().over(window))
df1.display()

In [0]:
window = Window.partitionBy(col("year")).orderBy(desc(col("total_amount")))
df1 = df1.withColumn("rank", rank().over(window))
df1.display()

In [0]:
window = Window.partitionBy(col("year")).orderBy(desc(col("total_amount")))
df1 = df1.withColumn("row_number", row_number().over(window))
df1.display()

### Data Writing

In [0]:
df.write.format("delta").mode("overwrite").save("abfss://silver@storageaccnitin.dfs.core.windows.net/orders")

In [0]:
%sql
create table if not exists dbcatalog.silver.orders_silver using delta
location 'abfss://silver@storageaccnitin.dfs.core.windows.net/orders'

In [0]:
%sql
select * from dbcatalog.silver.orders_silver